# ⚗ MIDAS LAB 3D v3 — 12 modelos, agora com ARIMA e Deep Learning (LSTM global)

> **Versão:** 3D-3.0 | **Horizonte:** 3 dias úteis | **Universo:** ações B3 ≤ R$ 3,00

---

## 🧬 O que foi absorvido dos 3 notebooks enviados (e o que foi adaptado)

| Notebook | O que ele faz | O que entrou no Midas — e por quê desse jeito |
|---|---|---|
| `stock-market-forecasting-arima` | `auto_arima` num único ativo dos EUA | Entrou o **modelo ARIMA** como 11º competidor — implementado como **AR(5) sobre log-retornos** em numpy puro. Rodar `auto_arima` dentro do walk-forward (60 validações × 5 janelas × ~50 ações) levaria horas para ganho marginal; o núcleo autorregressivo é preservado |
| `stock-market-analysis-prediction-using-lstm` | LSTM clássica prevendo o fechamento de 1 ação com 60 dias de lookback | Entrou a **LSTM** — mas **global**: uma única rede treinada em TODAS as ações do universo ao mesmo tempo. Treinar uma rede por ação com ~500 pregões é decoreba garantida (overfit); a versão global aprende padrões que se repetem entre ações |
| `lstm-model-with-market-and-news-data` (Two Sigma) | LSTM com dados de mercado + notícias, saída sigmoide de confiança | Entraram **duas ideias**: (1) a saída em **confiança de direção** (sigmoide → `2p−1`), que casa perfeitamente com o funil de convicção da v2; (2) **contexto de mercado** como feature (retorno diário do IBOV). O feed de **notícias** da Two Sigma é proprietário — não existe equivalente gratuito e confiável para penny stocks da B3, então essa parte **não** foi replicada (dizer isso é mais honesto que fingir) |

## 🔒 A regra de ouro do LSTM aqui (anti-leakage)

A rede é treinada **uma única vez**, apenas com dados **anteriores a TODAS as janelas de avaliação** (validação do campeonato + Prova Real), e depois **congelada**. Ela disputa o campeonato nas mesmas 60 janelas walk-forward que os outros 11 modelos, sob o mesmo LCB. Se ela perder o campeonato, perdeu — deep learning não ganha tratamento especial.

## 🗺️ Estrutura

| Célula | O que faz |
|---|---|
| 1 | Configuração |
| 2 | Universo ≤ R$ 3,00 + IBOV (regime + feature de contexto) |
| 3 | Os 12 modelos + validação walk-forward (LCB) |
| 4 | **LSTM global**: features → treino congelado no passado → confiança por ação/dia |
| 5 | Campeonato: 12 modelos × 5 janelas |
| 6 | Pipeline de hoje: previsões + funil de convicção + carteira |
| 7 | Backtest de 3 pregões (execução realista) |
| 8 | Prova Real: 30 janelas + Sharpe + drawdown + profit factor |
| 9 | Conclusões |

> Requisito novo: `tensorflow` (para a Célula 4). Sem ele, o notebook avisa e segue com os 11 modelos clássicos.
> ⚠️ *Sugestão algorítmica. Não constitui recomendação de investimento.*


In [ ]:
# ============================================================
# CÉLULA 1 — IMPORTAÇÕES E CONFIGURAÇÃO (MIDAS LAB 3D v3)
# ============================================================
# !pip install yfinance pandas numpy matplotlib openpyxl tensorflow -q

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# ── horizonte e validação ────────────────────────────────────
HORIZONTE      = 3        # dias úteis (D+1, D+2, D+3)
N_JANELAS_VAL  = 60       # janelas walk-forward (v1: 40) — mais amostras, LCB mais justo
JANELAS_TESTE  = [10, 21, 42, 63, 126]
DIAS_TREINO    = 63       # a Célula 3 sobrescreve com a janela campeã
Z_CONF         = 1.645    # 90% de confiança unilateral no Wilson Lower Bound
N_PROVA        = 30       # janelas da Prova Real (Célula 8)
N_VAL_PR       = 20       # janelas de validação por época na Prova Real

# ── Deep Learning: LSTM global (Célula 4) ───────────────────
USAR_LSTM      = True     # requer tensorflow; desativa sozinho se faltar
SEQ_LSTM       = 20       # dias de histórico por sequência de entrada
EPOCHS_LSTM    = 12       # épocas máximas (early stopping decide antes)

# ── universo (inalterado) ────────────────────────────────────
PRECO_MAX      = 3.00
PRECO_MIN      = 0.10
VOL_FIN_MIN    = 200_000
MIN_DIAS_DADOS = 200

# ── benchmark e custo ────────────────────────────────────────
SELIC_ANUAL    = 15.00    # % a.a. — ATUALIZE
SELIC_3D       = ((1 + SELIC_ANUAL/100) ** (HORIZONTE/252) - 1) * 100
CUSTO_OPER     = 1.00     # % ida-e-volta — agora DESCONTADO do resultado simulado
THRESHOLD      = SELIC_3D + CUSTO_OPER

# ── NOVOS filtros de convicção (coração da v2) ───────────────
SINAL_RUIDO_MIN = 0.50    # Δ previsto precisa ser ≥ 50% do movimento típico de 3d
CONSENSO_MIN    = 60.0    # % mínimo dos modelos concordando com a direção do campeão
USAR_REGIME     = True    # IBOV abaixo da média 50d → rebaixa COMPRAR p/ OBSERVAR
ENTRADA_ABERTURA = True   # simulações entram na ABERTURA de D+1 (realista)

# ── carteira por convicção (substitui o rank linear do JPX) ──
TOP_N          = 10
PESO_MAX       = 25.0     # teto de % por ação (diversificação forçada)

# ── identidade visual ────────────────────────────────────────
COR_FUNDO, COR_PAINEL, COR_OURO = '#0A0A0A', '#1A1A1A', '#D4AF37'

def estilo_eixo(ax, titulo='', xlabel='Data', ylabel='Preço (R$)'):
    ax.set_title(titulo, color='white', fontsize=12, pad=12)
    ax.set_xlabel(xlabel, color='white'); ax.set_ylabel(ylabel, color='white')
    ax.tick_params(colors='white')
    ax.grid(color='#2C2C2C', linestyle='--', linewidth=0.5)
    ax.set_facecolor(COR_PAINEL)

print('✅ MIDAS LAB 3D v3 — ambiente configurado')
print(f'   Horizonte            : {HORIZONTE} dias úteis | validação: {N_JANELAS_VAL} janelas')
print(f'   Universo             : R$ {PRECO_MIN:.2f}–{PRECO_MAX:.2f} | vol. fin. ≥ R$ {VOL_FIN_MIN:,.0f}/dia')
print(f'   Threshold de compra  : {THRESHOLD:.2f}% (Selic {SELIC_3D:.3f}% + custo {CUSTO_OPER:.2f}%)')
print(f'   Filtros de convicção : sinal/ruído ≥ {SINAL_RUIDO_MIN} | consenso ≥ {CONSENSO_MIN:.0f}% | regime IBOV: {"ON" if USAR_REGIME else "OFF"}')
print(f'   Execução simulada    : {"abertura de D+1, custo descontado" if ENTRADA_ABERTURA else "fechamento do sinal (v1)"}')
print(f'   Deep learning        : LSTM global {"ativada" if USAR_LSTM else "desativada"} '
      f'(seq={SEQ_LSTM}d, treino congelado no passado)')


---
## 🔬 Célula 2 — Universo ≤ R$ 3,00 + regime do IBOV

Igual à v1 (filtro dinâmico por preço atual + liquidez), com duas adições: agora guardamos também o preço de **abertura** (para a execução realista) e baixamos o **IBOV** para o filtro de regime — em mercado caindo, penny stock cai mais.

Na v3 o IBOV também vira **feature de contexto** do LSTM (retorno diário do índice).

In [ ]:
tickers_b3 = [
    'ABEV3.SA','ASAI3.SA','AZUL4.SA','B3SA3.SA','BBAS3.SA',
    'BBDC3.SA','BBDC4.SA','BBSE3.SA','BEEF3.SA','BHIA3.SA',
    'BPAC11.SA','BRAP4.SA','BRKM5.SA','BRFS3.SA','CBAV3.SA',
    'CCRO3.SA','CMIN3.SA','CMIG4.SA','COGN3.SA','CPFE3.SA',
    'CPLE3.SA','CPLE6.SA','CRFB3.SA','CSAN3.SA','CSNA3.SA',
    'CVCB3.SA','CXSE3.SA','CYRE3.SA','DXCO3.SA','ECOR3.SA',
    'EGIE3.SA','ELET3.SA','ELET6.SA','EMBR3.SA','EMBJ3.SA',
    'ENEV3.SA','ENGI11.SA','EQTL3.SA','FLRY3.SA','GGBR3.SA',
    'GGBR4.SA','GOAU3.SA','GOAU4.SA','GMAT3.SA','HAPV3.SA',
    'HYPE3.SA','IGTI11.SA','IRBR3.SA','ITSA4.SA','ITUB3.SA',
    'ITUB4.SA','JBSS3.SA','JHSF3.SA','KLBN11.SA','KLBN4.SA',
    'LREN3.SA','LWSA3.SA','MGLU3.SA','MOVI3.SA','MRFG3.SA',
    'MRVE3.SA','MULT3.SA','NATU3.SA','NTCO3.SA','PETR3.SA',
    'PETR4.SA','PRIO3.SA','PSSA3.SA','QUAL3.SA','RADL3.SA',
    'RAIL3.SA','RAIZ4.SA','RDOR3.SA','RENT3.SA','SANB11.SA',
    'SBSP3.SA','SLCE3.SA','SMFT3.SA','SUZB3.SA','TAEE11.SA',
    'TIMS3.SA','TOTS3.SA','UGPA3.SA','USIM3.SA','USIM5.SA',
    'VALE3.SA','VAMO3.SA','VBBR3.SA','VIVT3.SA','WEGE3.SA',
    'YDUQ3.SA','AGRO3.SA','ALOS3.SA','ALUP11.SA','AMAR3.SA',
    'ANIM3.SA','ARML3.SA','ARZZ3.SA','AURE3.SA','AUAU3.SA',
    'AXIA3.SA','AZZA3.SA','BPAN4.SA','BRAV3.SA','BRML3.SA',
    'BRPR3.SA','BRSR6.SA','CAML3.SA','CASH3.SA','CEAB3.SA',
    'CESP6.SA','CGAS5.SA','CGRA4.SA','CMIG3.SA','CSMG3.SA',
    'CURY3.SA','DESK3.SA','DIRR3.SA','DMVF3.SA','EVEN3.SA',
    'EZTC3.SA','FESA4.SA','FIQE3.SA','GGPS3.SA','GRND3.SA',
    'HBSA3.SA','INTB3.SA','ISAE4.SA','JSLG3.SA','KEPL3.SA',
    'LAVV3.SA','LEVE3.SA','LIGT3.SA','LJQQ3.SA','LOG3.SA',
    'LOGN3.SA','MBRF3.SA','MDIA3.SA','MELK3.SA','MILS3.SA',
    'MOTV3.SA','MYPK3.SA','NEOE3.SA','ONCO3.SA','ODPV3.SA',
    'OIBR3.SA','PCAR3.SA','PGMN3.SA','PINE4.SA','PLPL3.SA',
    'PNVL3.SA','POMO4.SA','POSI3.SA','RAPT4.SA','RCSL4.SA',
    'RECV3.SA','RLOG3.SA','ROMI3.SA','RRRP3.SA','SAPR11.SA',
    'SAUD3.SA','SBFG3.SA','SEER3.SA','SEQL3.SA','SIMH3.SA',
    'SMAG3.SA','SMTO3.SA','SOMA3.SA','SQIA3.SA','STBP3.SA',
    'SULA11.SA','TASA4.SA','TEND3.SA','TFCO4.SA','TRPL4.SA',
    'TTEN3.SA','TUPY3.SA','UCAS3.SA','UNIP6.SA','VIVA3.SA',
    'VLID3.SA','VULC3.SA','WEST3.SA','WIZC3.SA',
    'AALR3.SA','ABCB4.SA','AERI3.SA','AFLT3.SA','AGXY3.SA',
    'AGTE3.SA','AHEB3.SA','ALPA4.SA','ALSO3.SA','AMOB3.SA',
    'APER3.SA','ATMP3.SA','BAUH4.SA','BEES3.SA','BLAU3.SA',
    'BMGB4.SA','CAMB3.SA','CEEB3.SA','CEED4.SA','CLSA3.SA',
    'COCE5.SA','DASA3.SA','ENAT3.SA','ESPA3.SA','FHER3.SA',
    'FRTA3.SA','GFSA3.SA','HBOR3.SA','HETA4.SA','HGTX3.SA',
    'HMTL3.SA','JALL3.SA','JPSA3.SA','LAND3.SA','LPSB3.SA',
    'MDNE3.SA','MIXT3.SA','MNPR3.SA','NAIL3.SA','NCAB3.SA',
    'NEXP3.SA','NGRD3.SA','OFSA3.SA','ORVR3.SA','PARD3.SA',
    'PDGR3.SA','PETZ3.SA','PMAM3.SA','PTBL3.SA','RPMG3.SA',
    'RSUL4.SA','SAPR3.SA','SCAR3.SA','SOJA3.SA','SYNE3.SA',
    'TEKA4.SA','TGMA3.SA','TPVT3.SA','UPAR3.SA','VERS3.SA',
    'VIIA3.SA','XBXA3.SA',
]
tickers_b3 = list(dict.fromkeys(tickers_b3))
print(f'Lista B3 completa: {len(tickers_b3)} tickers')

# ── download único (2 anos, em lotes) — agora com Open ───────
cache_precos = {}   # {ticker: DataFrame[Data, Open, Close, Volume]}
LOTE = 60
print('Baixando 2 anos de histórico em lotes...')
for i in range(0, len(tickers_b3), LOTE):
    lote = tickers_b3[i:i+LOTE]
    try:
        raw = yf.download(lote, period='2y', interval='1d', group_by='ticker',
                          auto_adjust=True, progress=False, threads=True)
    except Exception as e:
        print(f'  ⚠️ falha no lote {i//LOTE + 1}: {e}')
        continue
    for tk in lote:
        try:
            df_tk = (raw if len(lote) == 1 else raw[tk])[['Open', 'Close', 'Volume']].copy()
            df_tk = df_tk.dropna()
            if df_tk.empty:
                continue
            df_tk.index = pd.to_datetime(df_tk.index.date)
            df_tk = df_tk.reset_index()
            df_tk.columns = ['Data', 'Open', 'Close', 'Volume']
            if len(df_tk) >= MIN_DIAS_DADOS:
                cache_precos[tk] = df_tk
        except Exception:
            pass
    print(f'  ✔ {min(i+LOTE, len(tickers_b3))}/{len(tickers_b3)} | válidos: {len(cache_precos)}')

# ── filtro: preço atual ≤ R$ 3,00 + liquidez ─────────────────
universo = []
for tk, df_tk in cache_precos.items():
    preco   = float(df_tk['Close'].iloc[-1])
    vol_fin = float((df_tk['Close'] * df_tk['Volume']).tail(21).median())
    if PRECO_MIN <= preco <= PRECO_MAX and vol_fin >= VOL_FIN_MIN:
        universo.append({'Ticker': tk.replace('.SA', ''), 'yf': tk,
                         'Preço (R$)': round(preco, 2),
                         'Vol. Fin. 21d (R$)': round(vol_fin, 0)})

df_universo  = pd.DataFrame(universo)
if not df_universo.empty:
    df_universo = df_universo.sort_values('Preço (R$)').reset_index(drop=True)
tickers_alvo = df_universo['yf'].tolist() if not df_universo.empty else []
ULTIMO_PREGAO = max(df['Data'].iloc[-1] for df in cache_precos.values()) if cache_precos else None

print(f'\n🎯 UNIVERSO — {len(tickers_alvo)} ações entre R$ {PRECO_MIN:.2f} e R$ {PRECO_MAX:.2f} com liquidez')
if ULTIMO_PREGAO is not None:
    print(f'   Último pregão nos dados: {ULTIMO_PREGAO.date()}')
if df_universo.empty:
    print('⚠️ Nenhuma ação passou no filtro — afrouxe PRECO_MAX ou VOL_FIN_MIN na Célula 1.')
else:
    print(df_universo.drop(columns=['yf']).to_string(index=False))

# ── IBOV: regime de mercado (média móvel 50d) ─────────────────
ibov = None
if USAR_REGIME:
    try:
        ib = yf.Ticker('^BVSP').history(period='2y', interval='1d')[['Close']].dropna()
        ib.index = pd.to_datetime(ib.index.date)
        ib = ib.reset_index(); ib.columns = ['Data', 'Close']
        ib['SMA50'] = ib['Close'].rolling(50).mean()
        ibov = ib.dropna().reset_index(drop=True)
        REGIME_HOJE = bool(ibov['Close'].iloc[-1] > ibov['SMA50'].iloc[-1])
        print(f"\n📈 IBOV: {ibov['Close'].iloc[-1]:,.0f} | média 50d: {ibov['SMA50'].iloc[-1]:,.0f} "
              f"→ regime {'de ALTA ✅ (compras liberadas)' if REGIME_HOJE else 'de BAIXA ⚠️ (compras rebaixadas p/ observar)'}")
    except Exception as e:
        print(f'\n⚠️ Falha ao baixar IBOV ({e}) — filtro de regime DESATIVADO nesta execução.')
        USAR_REGIME, REGIME_HOJE = False, True
else:
    REGIME_HOJE = True

_ibov_datas = ibov['Data'].values if ibov is not None else None
def regime_em(data):
    # Regime de mercado vigente NA DATA do sinal (para backtest/prova real,
    # usando apenas informação disponível até aquele dia)
    if not USAR_REGIME or ibov is None:
        return True
    pos = int(np.searchsorted(_ibov_datas, np.datetime64(data), side='right')) - 1
    if pos < 0:
        return True
    return bool(ibov['Close'].iloc[pos] > ibov['SMA50'].iloc[pos])


# ── retorno diário do IBOV por data (feature de contexto do LSTM) ──
if ibov is not None:
    _ret_ibov = ibov['Close'].pct_change().fillna(0)
    IBOV_RET = dict(zip(ibov['Data'], _ret_ibov))
else:
    IBOV_RET = {}
    print('ℹ️ Sem IBOV → feature de contexto do LSTM será 0 (rede segue funcionando).')


---
## 🔬 Célula 3 — Os 12 modelos e a validação honesta (LCB)

Aos 10 modelos da v2 somam-se **ARIMA (AR-Retornos)** — autorregressão de ordem 5 sobre log-retornos, o núcleo do notebook de ARIMA em versão rápida — e o **LSTM Global** (treinado na Célula 4; aqui fica só o encaixe dele na validação). O LSTM não prevê preço diretamente: prevê a **probabilidade de alta em 3 dias** (ideia do notebook Two Sigma), convertida em preço por `p₀ × (1 + (2p−1) × movimento típico de 3d)`.

Tudo o mais é herdado da v2: walk-forward sem lookahead, Wilson Lower Bound, funil de convicção.


In [ ]:
# ============================================================
# CÉLULA 3 — OS 12 MODELOS + VALIDAÇÃO WALK-FORWARD (LCB)
# ============================================================

MODELOS_CLASSICOS = ['Regressão Linear', 'Regressão Ponderada', 'Drift', 'Momentum',
                     'Reversão à Média', 'Z-Score Reversão', 'RSI-2', 'Média Modal',
                     'Holt', 'ARIMA (AR-Retornos)']
MODELOS = MODELOS_CLASSICOS + ['LSTM Global', 'Ensemble']

# ── modelos ──────────────────────────────────────────────────
def _holt(closes, h, alpha=0.5, beta=0.3):
    l = float(closes[0]); b = float(closes[1] - closes[0])
    for x in closes[1:]:
        l_ant = l
        l = alpha * float(x) + (1 - alpha) * (l + b)
        b = beta * (l - l_ant) + (1 - beta) * b
    return l + h * b

def media_modal(closes):
    s = pd.Series(np.round(np.asarray(closes, dtype=float), 2))
    modas = s.mode()
    if modas.empty:
        return float(s.median())
    p0 = float(closes[-1])
    return float(modas.iloc[(modas - p0).abs().values.argmin()])

def _rsi(closes, periodo=2):
    d = np.diff(np.asarray(closes, dtype=float)[-(periodo + 1):])
    ganho, perda = d[d > 0].sum(), -d[d < 0].sum()
    if perda == 0:
        return 100.0 if ganho > 0 else 50.0
    return 100.0 - 100.0 / (1.0 + ganho / perda)

def vol_movimento_h(closes, h=HORIZONTE):
    # movimento típico (mediana absoluta) em h dias, em %
    closes = np.asarray(closes, dtype=float)
    if len(closes) <= h:
        return 0.0
    r = closes[h:] / closes[:-h] - 1
    return float(np.median(np.abs(r))) * 100

def preco_de_confianca(p0, conf, treino, h=HORIZONTE):
    # Two Sigma: sigmoide -> confiança 2p-1 em [-1,1]; magnitude = movimento típico de 3d
    mov = max(vol_movimento_h(treino, HORIZONTE) / 100, 1e-4)
    return float(p0 * (1 + (2 * float(conf) - 1) * mov * h / HORIZONTE))

def prever(metodo, closes, h):
    closes = np.asarray(closes, dtype=float)
    n, p0 = len(closes), float(closes[-1])
    if n < 3:
        return p0
    t = np.arange(n, dtype=float)
    if metodo == 'Regressão Linear':
        b1, b0 = np.polyfit(t, closes, 1)          # ~10x mais rápido que sklearn, mesmo resultado
        return float(b0 + b1 * (n - 1 + h))
    if metodo == 'Regressão Ponderada':
        halflife = max(5.0, n / 4.0)
        w = 0.5 ** ((n - 1 - t) / halflife)        # dias recentes pesam mais
        b1, b0 = np.polyfit(t, closes, 1, w=np.sqrt(w))
        return float(b0 + b1 * (n - 1 + h))
    if metodo == 'Drift':
        return float(p0 + (closes[-1] - closes[0]) / max(n - 1, 1) * h)
    if metodo == 'Momentum':
        k = min(10, n - 1)
        roc = closes[-1] / closes[-1 - k] - 1 if closes[-1 - k] > 0 else 0.0
        return float(p0 * (1 + roc * h / max(k, 1)))
    if metodo == 'Reversão à Média':
        return float(closes[-min(21, n):].mean())
    if metodo == 'Z-Score Reversão':
        k = min(20, n)
        mu, sd = float(closes[-k:].mean()), float(closes[-k:].std())
        if sd <= 0:
            return p0
        z = (p0 - mu) / sd
        return float(p0 - 0.15 * h * z * sd)       # reversão parcial (~15%/dia do desvio)
    if metodo == 'RSI-2':
        rsi = _rsi(closes, 2)
        mov = max(vol_movimento_h(closes, h) / 100, 0.001)
        return float(p0 * (1 + mov * (50.0 - rsi) / 50.0))
    if metodo == 'ARIMA (AR-Retornos)':
        # ARIMA(5,1,0) prático: AR(5) sobre log-retornos, em numpy puro.
        # (auto_arima por janela seria ~1000x mais lento p/ ganho marginal)
        r = np.diff(np.log(np.maximum(closes, 1e-6)))
        p_ar = 5
        if len(r) < p_ar + 8:
            return p0
        Xl = np.column_stack([r[p_ar - 1 - k: len(r) - 1 - k] for k in range(p_ar)])
        yl = r[p_ar:]
        try:
            coef, *_ = np.linalg.lstsq(np.column_stack([np.ones(len(yl)), Xl]), yl, rcond=None)
        except Exception:
            return p0
        hist = list(r[-p_ar:])
        acum = 0.0
        for _ in range(h):
            nxt = float(np.clip(coef[0] + np.dot(coef[1:], hist[::-1][:p_ar]), -0.25, 0.25))
            acum += nxt
            hist.append(nxt)
        return float(p0 * np.exp(acum))
    if metodo == 'LSTM Global':
        return p0     # tratado fora do prever(): usa a confiança pré-computada (Célula 4)
    if metodo == 'Média Modal':
        return media_modal(closes)
    if metodo == 'Holt':
        return float(_holt(closes, h))
    if metodo == 'Ensemble':
        # mediana dos clássicos (o LSTM entra no consenso, não no ensemble)
        return float(np.median([prever(m, closes, h) for m in MODELOS_CLASSICOS]))
    return p0

# ── estatística honesta: Wilson Lower Bound ──────────────────
def wilson_lcb(acertos, n, z=Z_CONF):
    # "Qual o acerto MÍNIMO compatível com esses dados, com 90% de confiança?"
    if n == 0:
        return 0.0
    p = acertos / n
    den    = 1 + z * z / n
    centro = p + z * z / (2 * n)
    ajuste = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return max(0.0, (centro - ajuste) / den) * 100

def validar_walk_forward(closes, janela, metodo, h=HORIZONTE, n_val=N_JANELAS_VAL, conf=None):
    # Anti-leakage: cada previsão treina SÓ com dados anteriores à janela prevista
    closes = np.asarray(closes, dtype=float)
    n = len(closes)
    hits, erros = 0, []
    total = 0
    for i in range(n_val):
        fim = n - h * (n_val - i)
        ini = fim - janela
        alvo_idx = fim - 1 + h
        if ini < 0 or alvo_idx > n - 1:
            continue
        p0, alvo = closes[fim - 1], closes[alvo_idx]
        if p0 <= 0 or alvo <= 0:
            continue
        if metodo == 'LSTM Global':
            if conf is None or fim - 1 >= len(conf) or not np.isfinite(conf[fim - 1]):
                continue
            pred = preco_de_confianca(p0, conf[fim - 1], closes[ini:fim], h)
        else:
            pred = prever(metodo, closes[ini:fim], h)
        hits  += int(np.sign(pred - p0) == np.sign(alvo - p0))
        erros.append(abs(alvo - pred) / alvo * 100)
        total += 1
    if total == 0:
        return None
    return {'acerto': hits / total * 100,
            'lcb'   : wilson_lcb(hits, total),
            'mape'  : float(np.mean(erros)),
            'n'     : total}

# ── utilitários compartilhados (células 4–6) ─────────────────
def classificar(lcb):
    # limiares sobre o LCB (n=60: LCB 52 ≈ 63% bruto; LCB 47 ≈ 57,5% bruto)
    if lcb >= 52: return 'Muito Previsível'
    if lcb >= 50: return 'Previsibilidade Moderada'
    if lcb >= 47: return 'Pouco Previsível'
    return 'Não Previsível'

def consenso_direcional(closes_janela, dir_campeao, h=HORIZONTE, conf_val=None):
    # % dos modelos-base que concordam com a direção do campeão
    p0 = float(closes_janela[-1])
    dirs = [np.sign(prever(m, closes_janela, h) - p0) for m in MODELOS_CLASSICOS]
    if conf_val is not None and np.isfinite(conf_val):
        dirs.append(np.sign(float(conf_val) - 0.5))
    dirs = [d for d in dirs if d != 0]
    if not dirs or dir_campeao == 0:
        return 50.0
    return float(np.mean([d == dir_campeao for d in dirs]) * 100)

def sugerir(classe, delta_pct, consenso, snr, regime_ok=True):
    # Funil da v2: retorno mínimo → classe → convicção (consenso+ruído) → regime
    if delta_pct < THRESHOLD or classe == 'Não Previsível':
        return '❌ NÃO INDICADO'
    forte = classe in ('Muito Previsível', 'Previsibilidade Moderada')
    if forte and consenso >= CONSENSO_MIN and snr >= SINAL_RUIDO_MIN and regime_ok:
        return '✅ COMPRAR'
    return '⚠️ OBSERVAR'

def escolher_campeao(closes, janela, n_val=N_JANELAS_VAL, conf=None):
    # Campeão = maior LCB (desempate: menor MAPE) — não o maior acerto bruto
    melhor = None
    for m in MODELOS:
        if m == 'LSTM Global' and conf is None:
            continue
        r = validar_walk_forward(closes, janela, m, n_val=n_val, conf=conf)
        if r is None or r['n'] < max(10, n_val // 2):
            continue
        chave = (r['lcb'], -r['mape'])
        if melhor is None or chave > melhor['chave']:
            melhor = {'modelo': m, 'acerto': r['acerto'], 'lcb': r['lcb'],
                      'mape': r['mape'], 'n': r['n'], 'chave': chave}
    return melhor

def conviccao(lcb, consenso, snr, vol_ratio):
    # Multiplicador de convicção ∈ [0, ~2]: quanto confiar neste Δ previsto
    return ((lcb / 50.0)
            * (consenso / 100.0)
            * min(snr, 2.0) / 2.0
            * float(np.clip(vol_ratio, 0.8, 1.2)))

def pesos_conviccao(deltas_ajustados):
    # Kelly-lite: peso ∝ Δ ajustado, com teto PESO_MAX por ação.
    # Waterfall de conjunto fixo: quem bate no teto fica travado lá e o
    # restante é redistribuído proporcionalmente — sem oscilar.
    s = np.clip(np.asarray(deltas_ajustados, dtype=float), 0.05, None)
    n = len(s)
    if n == 0:
        return np.array([])
    if n * PESO_MAX < 100:            # teto inviável (poucas ações) → peso igual
        return np.round(np.full(n, 100.0 / n), 1)
    w = s / s.sum() * 100
    fixos = np.zeros(n, dtype=bool)
    for _ in range(n):
        exc = (w > PESO_MAX + 1e-9) & ~fixos
        if not exc.any():
            break
        fixos |= exc
        w[fixos] = PESO_MAX
        livres = ~fixos
        restante = 100 - PESO_MAX * fixos.sum()
        if livres.any():
            w[livres] = s[livres] / s[livres].sum() * max(restante, 0)
    return np.round(w, 1)

def estilizar_excel(writer, col_sugestao='Sugestão'):
    from openpyxl.styles import PatternFill, Font, Alignment
    from openpyxl.utils import get_column_letter
    for sn, ws in writer.sheets.items():
        headers = [c.value for c in ws[1]]
        idx_sug = headers.index(col_sugestao) if col_sugestao in headers else None
        for cell in ws[1]:
            cell.fill = PatternFill('solid', fgColor='0A0A0A')
            cell.font = Font(bold=True, color='D4AF37')
            cell.alignment = Alignment(horizontal='center', vertical='center')
        for row in ws.iter_rows(min_row=2):
            sug = row[idx_sug].value if idx_sug is not None else ''
            fg = ('0D3B1F' if sug == '✅ COMPRAR' else
                  '3B2D00' if sug == '⚠️ OBSERVAR' else
                  '1A0A0A' if sug == '❌ NÃO INDICADO' else '111111')
            for cell in row:
                cell.fill = PatternFill('solid', fgColor=fg)
                cell.font = Font(color='CCCCCC')
                cell.alignment = Alignment(horizontal='center', vertical='center')
        for col in ws.columns:
            ml = max((len(str(c.value or '')) for c in col), default=10)
            ws.column_dimensions[get_column_letter(col[0].column)].width = min(ml + 4, 30)
        ws.freeze_panes = 'A2'



---
## 🧠 Célula 4 — LSTM Global: uma rede para todo o universo

Cada dia de cada ação vira uma **sequência de 20 dias × 6 features**: retorno 1d, retorno 3d, volume 5d/21d, z-score 20d, RSI-2 e retorno do IBOV (o "contexto de mercado" herdado do Two Sigma). A rede (LSTM 32 → Dense 16 → sigmoide) aprende a responder uma única pergunta: **qual a probabilidade de esta ação fechar mais alta daqui a 3 pregões?**

**Corte temporal:** só entram no treino sequências cujo alvo cai **antes** de todas as janelas usadas depois (campeonato + Prova Real). Depois de treinada, a rede é congelada e apenas consultada — igual a qualquer outro competidor.


In [ ]:
# ============================================================
# CÉLULA 4 — LSTM GLOBAL (treino congelado no passado)
# ============================================================

LSTM_CONF = {}   # {ticker: array alinhado ao df — confiança P(alta em 3d) por dia}
N_FEAT = 6
# tudo avaliado depois (60 janelas do campeonato; 30 de prova + 20 de validação
# por época) precisa ficar FORA do treino:
CORTE_OOS = HORIZONTE * max(N_JANELAS_VAL, N_PROVA + N_VAL_PR) + HORIZONTE + 7

if USAR_LSTM:
    try:
        import tensorflow as tf
        from tensorflow import keras
        tf.get_logger().setLevel('ERROR')
    except Exception as e:
        print(f'⚠️ tensorflow indisponível ({e}) → LSTM desativado; seguimos com 11 modelos.')
        USAR_LSTM = False

def features_ticker(df_tk):
    c = df_tk['Close'].astype(float)
    v = df_tk['Volume'].astype(float)
    f = pd.DataFrame(index=df_tk.index)
    f['ret1'] = c.pct_change().clip(-0.15, 0.15).fillna(0)
    f['ret3'] = c.pct_change(HORIZONTE).clip(-0.30, 0.30).fillna(0)
    f['volr'] = (v.rolling(5).mean() / v.rolling(21).mean()).clip(0, 3).fillna(1) - 1
    m20, s20 = c.rolling(20).mean(), c.rolling(20).std()
    f['z20']  = ((c - m20) / s20.replace(0, np.nan)).clip(-3, 3).fillna(0)
    d = c.diff(); up = d.clip(lower=0); dn = (-d).clip(lower=0)
    g2, l2 = up.rolling(2).mean(), dn.rolling(2).mean()
    f['rsi2'] = (g2 / (g2 + l2)).fillna(0.5) - 0.5
    f['ibov'] = pd.Series(df_tk['Data'].map(IBOV_RET).values, index=df_tk.index).fillna(0).clip(-0.1, 0.1)
    return f.values.astype('float32')

if USAR_LSTM:
    np.random.seed(42); tf.random.set_seed(42)
    X_tr, y_tr, X_va, y_va, seqs = [], [], [], [], {}
    for tk in tickers_alvo:
        df_tk = cache_precos[tk]
        c = df_tk['Close'].values.astype(float)
        F = features_ticker(df_tk)
        n = len(F)
        LSTM_CONF[tk] = np.full(n, np.nan, dtype='float32')
        if n <= SEQ_LSTM + HORIZONTE:
            continue
        idx_t = np.arange(SEQ_LSTM - 1, n)                     # dias com sequência completa
        seq = np.stack([F[t + 1 - SEQ_LSTM: t + 1] for t in idx_t])
        seqs[tk] = (idx_t, seq)
        corte_tk = n - CORTE_OOS                               # início da zona intocável
        mask_tr = (idx_t + HORIZONTE) < corte_tk               # alvo 100% no passado
        if mask_tr.sum() < 40:
            continue
        alvo = (c[idx_t[mask_tr] + HORIZONTE] > c[idx_t[mask_tr]]).astype('float32')
        s_tr = seq[mask_tr]
        n_va = max(int(len(s_tr) * 0.15), 5)                   # val temporal: fim do treino
        X_tr.append(s_tr[:-n_va]); y_tr.append(alvo[:-n_va])
        X_va.append(s_tr[-n_va:]); y_va.append(alvo[-n_va:])

    if not X_tr:
        print('⚠️ Histórico insuficiente p/ treinar o LSTM → desativado.')
        USAR_LSTM = False
    else:
        Xtr = np.concatenate(X_tr); ytr = np.concatenate(y_tr)
        Xva = np.concatenate(X_va); yva = np.concatenate(y_va)
        print(f'Treino: {len(Xtr):,} sequências | validação temporal: {len(Xva):,} | '
              f'zona fora do treino: últimos {CORTE_OOS} pregões de cada ação')

        modelo_lstm = keras.Sequential([
            keras.layers.Input(shape=(SEQ_LSTM, N_FEAT)),
            keras.layers.LSTM(32),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(16, activation='relu'),
            keras.layers.Dense(1, activation='sigmoid'),
        ])
        modelo_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
        es = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True,
                                           monitor='val_loss')
        hist = modelo_lstm.fit(Xtr, ytr, validation_data=(Xva, yva),
                               epochs=EPOCHS_LSTM, batch_size=256, verbose=0, callbacks=[es])
        acc_va = float(hist.history['val_accuracy'][np.argmin(hist.history['val_loss'])])
        print(f'✅ LSTM treinada e CONGELADA | acurácia direcional na validação (pré-corte): '
              f'{acc_va*100:.1f}% | épocas: {len(hist.history["val_loss"])}')

        # consulta em lote: confiança para TODOS os dias com sequência completa
        confs_oos = []
        for tk, (idx_t, seq) in seqs.items():
            p = modelo_lstm.predict(seq, verbose=0)[:, 0]
            LSTM_CONF[tk][idx_t] = p
            corte_tk = len(LSTM_CONF[tk]) - CORTE_OOS
            confs_oos.append(p[idx_t >= corte_tk])
        confs_oos = np.concatenate(confs_oos) if confs_oos else np.array([])

        # histograma da confiança fora-do-treino (estilo do notebook Two Sigma)
        if len(confs_oos):
            fig, ax = plt.subplots(figsize=(11, 4))
            ax.hist(confs_oos * 2 - 1, bins=40, color=COR_OURO, edgecolor='#333333')
            ax.axvline(0, color='white', lw=0.8, ls=':')
            estilo_eixo(ax, 'LSTM — confiança de direção fora do treino (2p−1; −1 = queda certa, +1 = alta certa)',
                        xlabel='Confiança', ylabel='Nº de previsões')
            fig.patch.set_facecolor(COR_FUNDO)
            plt.tight_layout(); plt.show()
            print(f'Confiança média |2p−1| fora do treino: {np.mean(np.abs(confs_oos*2-1)):.3f} '
                  f'(perto de 0 = rede insegura; o campeonato dirá se ela merece confiança)')
else:
    print('Campeonato seguirá apenas com os 11 modelos clássicos (+ Ensemble).')


---
## 🏆 Célula 5 — Campeonato: 12 modelos × 5 janelas

Mesmo torneio da v2, agora com ARIMA e LSTM na chave. O LSTM é consultado pela confiança pré-computada — mesmas janelas, mesmo LCB, zero privilégio. Se `tensorflow` não estiver instalado, ele simplesmente não pontua.


In [ ]:
# ═════════════════════════ CAMPEONATO ═════════════════════════
placar = {(m, j): {'lcb': [], 'acerto': [], 'mape': []} for m in MODELOS for j in JANELAS_TESTE}
print(f'Campeonato: {len(tickers_alvo)} ações × {len(JANELAS_TESTE)} janelas × {len(MODELOS)} modelos...\n')

for i, tk in enumerate(tickers_alvo, 1):
    closes = cache_precos[tk]['Close'].values
    conf_tk = LSTM_CONF.get(tk)
    for j in JANELAS_TESTE:
        for m in MODELOS:
            r = validar_walk_forward(closes, j, m, conf=conf_tk)
            if r and r['n'] >= 30:
                placar[(m, j)]['lcb'].append(r['lcb'])
                placar[(m, j)]['acerto'].append(r['acerto'])
                placar[(m, j)]['mape'].append(r['mape'])
    if i % 10 == 0 or i == len(tickers_alvo):
        print(f'  ✔ {i}/{len(tickers_alvo)}')

linhas = []
for (m, j), v in placar.items():
    if not v['lcb']:
        continue
    linhas.append({'Modelo': m, 'Janela (d)': j, 'Ações': len(v['lcb']),
                   'Acerto Bruto (%)': round(float(np.mean(v['acerto'])), 1),
                   'LCB Acerto (%)'  : round(float(np.mean(v['lcb'])), 1),
                   'MAPE D+3 (%)'    : round(float(np.mean(v['mape'])), 2)})

df_camp = pd.DataFrame(linhas)
mape_v  = df_camp['MAPE D+3 (%)'].astype(float)
mape_n  = 100 - (mape_v - mape_v.min()) / (mape_v.max() - mape_v.min() + 1e-9) * 100
df_camp['⭐ Score'] = (0.5 * mape_n + 0.5 * df_camp['LCB Acerto (%)'].astype(float)).round(1)
df_camp = df_camp.sort_values('⭐ Score', ascending=False).reset_index(drop=True)

MELHOR_MODELO_GLOBAL = df_camp.iloc[0]['Modelo']
MELHOR_JANELA        = int(df_camp.iloc[0]['Janela (d)'])

print('\n' + '═' * 78)
print(f'  CAMPEONATO v3 — 12 MODELOS — ranking por LCB (acerto mínimo, 90% confiança)')
print('═' * 78)
print(df_camp.head(15).to_string(index=False))
print('═' * 78)
print(f'  🏆 Campeã global: {MELHOR_MODELO_GLOBAL} @ {MELHOR_JANELA}d '
      f'(a Célula 4 escolhe o campeão POR AÇÃO nesta janela)')

df_modelos = (df_camp.groupby('Modelo')
              .agg({'LCB Acerto (%)': 'mean', 'Acerto Bruto (%)': 'mean',
                    'MAPE D+3 (%)': 'mean', '⭐ Score': 'mean'})
              .round(1).sort_values('⭐ Score', ascending=False))
print('\n── Ranking médio por MODELO (todas as janelas) ──')
print(df_modelos.to_string())

top10 = df_camp.head(10).iloc[::-1]
rotulos = top10['Modelo'] + ' @ ' + top10['Janela (d)'].astype(str) + 'd'
fig, ax = plt.subplots(figsize=(12, 6))
cores = [COR_OURO if (r['Modelo'] == MELHOR_MODELO_GLOBAL and int(r['Janela (d)']) == MELHOR_JANELA)
         else '#00CED1' for _, r in top10.iterrows()]
bars = ax.barh(rotulos, top10['⭐ Score'], color=cores, edgecolor='#333333')
for bar, val in zip(bars, top10['⭐ Score']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', color='white', fontsize=9)
estilo_eixo(ax, 'Top 10 combinações Modelo × Janela — Score (0.5·MAPE_norm + 0.5·LCB)',
            xlabel='⭐ Score', ylabel='')
fig.patch.set_facecolor(COR_FUNDO)
plt.tight_layout(); plt.show()


---
## 📊 Célula 6 — Pipeline de hoje: previsão + funil de convicção + carteira Kelly-lite

Para cada ação: campeão por LCB → previsão D+1..D+3 → e então o **funil da v2**:

`Δ ≥ threshold` → `classe previsível (LCB)` → `consenso ≥ 60%` → `sinal/ruído ≥ 0,5` → `regime IBOV ok` → **✅ COMPRAR**

Falhou em consenso, ruído ou regime (mas passou no resto)? Vira **⚠️ OBSERVAR** — o sinal existe, a convicção não.

A carteira final abandona o rank linear: **peso ∝ Δ ajustado pela convicção** (LCB × consenso × sinal/ruído × volume), com teto de 25% por ação.


In [ ]:
# ============================================================
# CÉLULA 6 — PIPELINE FINAL v3: PREVISÕES DE HOJE COM CONVICÇÃO
# ============================================================

PASTA = 'resultado_3d'
os.makedirs(PASTA, exist_ok=True)
resultados = []

def rotulos_projecao_3d(ax, datas_fut, y_fut, y_max_ref):
    marcos = [(0, 'D+1', '#F0D060'), (1, 'D+2', '#D4AF37'), (2, 'D+3', '#A08020')]
    for idx, label, cor in marcos:
        if idx >= len(datas_fut):
            continue
        x, y = datas_fut[idx], float(y_fut[idx])
        ax.scatter(x, y, color=cor, s=45, zorder=5)
        y_lbl = min(y + y_max_ref * 0.05, y_max_ref * 1.04)
        ax.annotate(f'{label}\nR${y:.2f}', xy=(x, y), xytext=(x, y_lbl),
                    textcoords='data', ha='center', fontsize=7.5, color=cor,
                    rotation=90, va='bottom',
                    arrowprops=dict(arrowstyle='-', color='#555555', lw=0.7))

print(f'Processando {len(tickers_alvo)} ações (janela campeã = {MELHOR_JANELA}d) | '
      f'regime IBOV hoje: {"ALTA ✅" if REGIME_HOJE else "BAIXA ⚠️"}\n')

for i, tk in enumerate(tickers_alvo, 1):
    try:
        df_tk  = cache_precos[tk]
        closes = df_tk['Close'].values.astype(float)
        datas  = df_tk['Data']
        conf_tk = LSTM_CONF.get(tk)
        cval_hoje = (float(conf_tk[-1]) if conf_tk is not None and len(conf_tk)
                     and np.isfinite(conf_tk[-1]) else None)

        camp = escolher_campeao(closes, MELHOR_JANELA, conf=conf_tk)
        if camp is None:
            continue
        classe = classificar(camp['lcb'])

        treino = closes[-MELHOR_JANELA:]
        preco_atual = float(closes[-1])
        if camp['modelo'] == 'LSTM Global' and cval_hoje is not None:
            y_fut = np.array([preco_de_confianca(preco_atual, cval_hoje, treino, h)
                              for h in range(1, HORIZONTE + 1)])
        else:
            y_fut = np.array([prever(camp['modelo'], treino, h) for h in range(1, HORIZONTE + 1)])
        datas_fut = pd.bdate_range(start=datas.iloc[-1] + timedelta(days=1), periods=HORIZONTE)

        mm        = media_modal(treino)
        preco_d3  = float(y_fut[-1])
        delta_pct = (preco_d3 - preco_atual) / preco_atual * 100
        direcao   = 'ALTA' if preco_d3 > preco_atual else 'QUEDA'

        # ── funil de convicção v2 ────────────────────────────
        vol3d    = vol_movimento_h(treino)                       # movimento típico de 3d (%)
        snr      = abs(delta_pct) / vol3d if vol3d > 0 else 0.0  # sinal/ruído
        consenso = consenso_direcional(treino, np.sign(preco_d3 - preco_atual), conf_val=cval_hoje)
        vol_ratio = float(df_tk['Volume'].tail(5).mean() /
                          max(df_tk['Volume'].tail(21).mean(), 1))
        conv      = conviccao(camp['lcb'], consenso, snr, vol_ratio)
        delta_adj = delta_pct * conv
        sugestao  = sugerir(classe, delta_pct, consenso, snr, regime_ok=REGIME_HOJE)
        vol_fin   = float((df_tk['Close'] * df_tk['Volume']).tail(21).median())

        # ── gráfico individual ───────────────────────────────
        vis = df_tk.tail(min(MELHOR_JANELA, 63))
        y_max_ref  = max(float(vis['Close'].max()), float(y_fut.max()), mm)
        margem_sup = y_max_ref * 1.22
        fig, ax = plt.subplots(figsize=(13, 5.5))
        ax.set_ylim(min(float(vis['Close'].min()), float(y_fut.min()), mm) * 0.90, margem_sup)
        ax.plot(vis['Data'], vis['Close'], color=COR_OURO, linewidth=1.4,
                label=f'Real (janela {MELHOR_JANELA}d)')
        ax.axhline(mm, color='#F0D060', linewidth=1.1, linestyle=':',
                   label=f'Média modal: R$ {mm:.2f}')
        ax.plot([vis['Data'].iloc[-1], *datas_fut], [preco_atual, *y_fut],
                color='#00CED1', linewidth=2.0, linestyle='--',
                label=f'Projeção 3d ({camp["modelo"]})')
        rotulos_projecao_3d(ax, datas_fut, y_fut, margem_sup)
        ax.axvline(vis['Data'].iloc[-1], color='#555555', linewidth=0.8, linestyle=':')
        info = (f'{camp["modelo"]}  |  Acerto: {camp["acerto"]:.0f}% (LCB {camp["lcb"]:.0f}%, '
                f'{camp["n"]} jan.)  |  Consenso: {consenso:.0f}%  |  S/R: {snr:.2f}  |  '
                f'Δ D+3: {delta_pct:+.2f}%  |  {direcao}  |  {sugestao}')
        ax.text(0.01, 0.02, info, transform=ax.transAxes, color='#AAAAAA',
                fontsize=7.5, va='bottom')
        tl = tk.replace('.SA', '')
        estilo_eixo(ax, f'{tl} — Projeção 3 dias | preço atual R$ {preco_atual:.2f}')
        ax.legend(facecolor=COR_PAINEL, labelcolor='white', fontsize=8, loc='upper left')
        fig.patch.set_facecolor(COR_FUNDO)
        plt.tight_layout()
        plt.savefig(f'{PASTA}/{tl}.png', dpi=100, bbox_inches='tight', facecolor=COR_FUNDO)
        plt.close()

        resultados.append({
            'Ticker'             : tl,
            'Preço Atual (R$)'   : round(preco_atual, 2),
            'Modelo Campeão'     : camp['modelo'],
            'Acerto Bruto (%)'   : round(camp['acerto'], 1),
            'LCB Acerto (%)'     : round(camp['lcb'], 1),
            'Previsibilidade'    : classe,
            'Consenso (%)'       : round(consenso, 0),
            'Conf. LSTM (%)'     : (round(cval_hoje * 100, 0) if cval_hoje is not None else None),
            'Vol 3d Típica (%)'  : round(vol3d, 2),
            'Sinal/Ruído'        : round(snr, 2),
            'Volume 5d/21d'      : round(vol_ratio, 2),
            'Média Modal (R$)'   : round(mm, 2),
            'Preço D+1 (R$)'     : round(float(y_fut[0]), 2),
            'Preço D+2 (R$)'     : round(float(y_fut[1]), 2),
            'Preço D+3 (R$)'     : round(float(y_fut[2]), 2),
            'Δ D+3 vs Hoje (%)'  : round(delta_pct, 2),
            'Δ Ajustado (%)'     : round(delta_adj, 2),
            'Direção'            : direcao,
            'Vol. Fin. 21d (R$)' : round(vol_fin, 0),
            'Sugestão'           : sugestao,
        })
    except Exception:
        pass
    if i % 10 == 0 or i == len(tickers_alvo):
        print(f'  ✔ {i}/{len(tickers_alvo)} | processadas: {len(resultados)}')

df_final = pd.DataFrame(resultados)
print(f'\n✅ {len(df_final)} ações processadas')

if not df_final.empty:
    ordem = {'✅ COMPRAR': 0, '⚠️ OBSERVAR': 1, '❌ NÃO INDICADO': 2}
    df_final['_ord'] = df_final['Sugestão'].map(ordem)
    df_final = (df_final.sort_values(['_ord', 'Δ Ajustado (%)'], ascending=[True, False])
                        .drop(columns=['_ord']).reset_index(drop=True))

    # ── carteira Kelly-lite: peso ∝ Δ ajustado pela convicção ─
    compras  = df_final[df_final['Sugestão'] == '✅ COMPRAR'].copy()
    carteira = compras.sort_values('Δ Ajustado (%)', ascending=False).head(TOP_N).copy()
    if not carteira.empty:
        carteira.insert(0, 'Rank', np.arange(1, len(carteira) + 1))
        carteira['Peso Sugerido (%)'] = pesos_conviccao(carteira['Δ Ajustado (%)'].values)

    excel_path = f'{PASTA}/Midas_Lab_3D_v3_Previsoes.xlsx'
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        df_final.to_excel(writer, sheet_name='Consolidado', index=False)
        df_final[df_final['Sugestão'] == '✅ COMPRAR'].to_excel(
            writer, sheet_name='Sugestões de Compra', index=False)
        df_final[df_final['Sugestão'] == '⚠️ OBSERVAR'].to_excel(
            writer, sheet_name='Observar', index=False)
        if not carteira.empty:
            cols_cart = ['Rank', 'Ticker', 'Preço Atual (R$)', 'Modelo Campeão',
                         'LCB Acerto (%)', 'Consenso (%)', 'Sinal/Ruído',
                         'Δ D+3 vs Hoje (%)', 'Δ Ajustado (%)',
                         'Peso Sugerido (%)', 'Sugestão']
            carteira[cols_cart].to_excel(writer, sheet_name='Carteira Convicção', index=False)
        estilizar_excel(writer)
    print(f'📊 Excel salvo em : {excel_path}')
    print(f'🖼️  Gráficos em   : /{PASTA}/')

    cols = ['Ticker', 'Preço Atual (R$)', 'Modelo Campeão', 'LCB Acerto (%)',
            'Consenso (%)', 'Sinal/Ruído', 'Δ D+3 vs Hoje (%)', 'Δ Ajustado (%)',
            'Direção', 'Sugestão']
    ref = f' (referência: {ULTIMO_PREGAO.date()})' if ULTIMO_PREGAO is not None else ''
    print(f'\n── Top 10 Sugestões — 3 dias úteis{ref} ──')
    top_comprar = df_final[df_final['Sugestão'] == '✅ COMPRAR']
    if top_comprar.empty:
        print('  Nenhuma ação passou no funil completo hoje — é o filtro fazendo o trabalho dele.')
        print('  Candidatas mais próximas (falharam em consenso, ruído ou regime):')
        print(df_final.head(10)[cols].to_string(index=False))
    else:
        print(top_comprar.head(10)[cols].to_string(index=False))

    if not carteira.empty:
        print(f'\n── 💼 Carteira por Convicção (teto {PESO_MAX:.0f}%/ação) ──')
        print('   Lembrete: os preços de entrada de referência são os da PRÓXIMA ABERTURA.')
        print(carteira[['Rank', 'Ticker', 'Preço Atual (R$)', 'Δ D+3 vs Hoje (%)',
                        'Δ Ajustado (%)', 'Peso Sugerido (%)']].to_string(index=False))


---
## 🧪 Célula 7 — Backtest com execução realista

Igual à v1 (esconde os últimos 3 pregões e roda o pipeline completo só com o passado), com duas doses de realidade:
- **Entrada na abertura de D+1** — quem roda o modelo depois do fechamento não compra no preço que acabou de ver;
- **Custo operacional descontado** do resultado (não só do threshold).

O acerto direcional continua medido fechamento-a-fechamento (é o que o modelo prevê); o **dinheiro** é medido do jeito que você operaria.


In [ ]:
# ============================================================
# CÉLULA 7 — BACKTEST v3: 3 PREGÕES COM EXECUÇÃO REALISTA
# ============================================================

PASTA_BT = 'backtest_3d'
os.makedirs(PASTA_BT, exist_ok=True)
INVESTIMENTO = 100_000
resultados_bt = []

print(f'Backtest de {HORIZONTE} pregões em {len(tickers_alvo)} ações...\n')

for i, tk in enumerate(tickers_alvo, 1):
    try:
        df_tk  = cache_precos[tk]
        closes = df_tk['Close'].values.astype(float)
        opens  = df_tk['Open'].values.astype(float)
        datas  = df_tk['Data']
        if len(closes) < MELHOR_JANELA + HORIZONTE * (N_JANELAS_VAL // 2) + HORIZONTE:
            continue

        passado = closes[:-HORIZONTE]
        holdout = closes[-HORIZONTE:]
        conf_tk = LSTM_CONF.get(tk)
        conf_pass = conf_tk[:-HORIZONTE] if conf_tk is not None and len(conf_tk) else None
        cval_bt = (float(conf_pass[-1]) if conf_pass is not None and len(conf_pass)
                   and np.isfinite(conf_pass[-1]) else None)
        camp = escolher_campeao(passado, MELHOR_JANELA, conf=conf_pass)
        if camp is None:
            continue
        classe = classificar(camp['lcb'])

        p0 = float(passado[-1])                       # fechamento do dia do sinal
        treino = passado[-MELHOR_JANELA:]
        if camp['modelo'] == 'LSTM Global' and cval_bt is not None:
            y_prev = np.array([preco_de_confianca(p0, cval_bt, treino, h)
                               for h in range(1, HORIZONTE + 1)])
        else:
            y_prev = np.array([prever(camp['modelo'], treino, h) for h in range(1, HORIZONTE + 1)])
        preco_prev_d3 = float(y_prev[-1])
        preco_real_d3 = float(holdout[-1])

        delta_prev = (preco_prev_d3 - p0) / p0 * 100
        delta_real = (preco_real_d3 - p0) / p0 * 100          # fech→fech (mede o MODELO)
        entrada_exec = float(opens[-HORIZONTE]) if ENTRADA_ABERTURA else p0
        delta_exec = (preco_real_d3 - entrada_exec) / entrada_exec * 100  # mede o BOLSO

        vol3d    = vol_movimento_h(treino)
        snr      = abs(delta_prev) / vol3d if vol3d > 0 else 0.0
        consenso = consenso_direcional(treino, np.sign(preco_prev_d3 - p0), conf_val=cval_bt)
        data_sinal = datas.iloc[len(passado) - 1]
        sugestao = sugerir(classe, delta_prev, consenso, snr,
                           regime_ok=regime_em(data_sinal))
        acertou = np.sign(delta_prev) == np.sign(delta_real)

        resultados_bt.append({
            'Ticker'             : tk.replace('.SA', ''),
            'Modelo Campeão'     : camp['modelo'],
            'LCB Acerto (%)'     : round(camp['lcb'], 1),
            'Consenso (%)'       : round(consenso, 0),
            'Sinal/Ruído'        : round(snr, 2),
            'Preço Sinal (R$)'   : round(p0, 2),
            'Entrada Exec. (R$)' : round(entrada_exec, 2),
            'Saída Real (R$)'    : round(preco_real_d3, 2),
            'Δ Previsto 3d (%)'  : round(delta_prev, 2),
            'Δ Real fech (%)'    : round(delta_real, 2),
            'Δ Executado (%)'    : round(delta_exec, 2),
            'Acertou Direção?'   : '✅ SIM' if acertou else '❌ NÃO',
            'Sugestão (3d atrás)': sugestao,
        })

        if sugestao == '✅ COMPRAR':
            vis = df_tk.tail(45)
            datas_hold = datas.iloc[-HORIZONTE:]
            y_max_bt = max(float(vis['Close'].max()), float(y_prev.max()))
            margem   = y_max_bt * 1.22
            fig, ax = plt.subplots(figsize=(13, 5.5))
            ax.set_ylim(float(vis['Close'].min()) * 0.85, margem)
            ax.plot(vis['Data'].iloc[:-HORIZONTE], vis['Close'].iloc[:-HORIZONTE],
                    color=COR_OURO, linewidth=1.3, label='Real (antes do sinal)')
            ax.plot([datas.iloc[-HORIZONTE-1], *datas_hold], [p0, *holdout],
                    color='#FFFFFF', linewidth=1.6, label='Real (3 pregões escondidos)')
            ax.plot([datas.iloc[-HORIZONTE-1], *datas_hold], [p0, *y_prev],
                    color='#FF8C00', linewidth=1.6, linestyle='--', label='Previsto')
            ax.scatter(datas_hold.iloc[0], entrada_exec, color='#00CED1', s=50, zorder=6,
                       label=f'Entrada real (abertura): R$ {entrada_exec:.2f}')
            ax.fill_between(datas_hold, holdout, y_prev, alpha=0.15,
                            color='#00FF88' if delta_exec >= 0 else '#FF4444')
            ax.axvline(datas.iloc[-HORIZONTE-1], color='#555555', lw=0.8, ls=':')
            info = (f'{camp["modelo"]}  |  Δ Prev: {delta_prev:+.2f}%  |  '
                    f'Δ Exec: {delta_exec:+.2f}% (bruto)  |  Acertou? {"✅" if acertou else "❌"}')
            ax.text(0.01, 0.02, info, transform=ax.transAxes, color='#AAAAAA',
                    fontsize=7.5, va='bottom')
            tl = tk.replace('.SA', '')
            estilo_eixo(ax, f'{tl} — Backtest: sinal {HORIZONTE} pregões atrás, entrada na abertura seguinte')
            ax.legend(facecolor=COR_PAINEL, labelcolor='white', fontsize=8)
            fig.patch.set_facecolor(COR_FUNDO)
            plt.tight_layout()
            plt.savefig(f'{PASTA_BT}/{tl}_backtest.png', dpi=100,
                        bbox_inches='tight', facecolor=COR_FUNDO)
            plt.close()
    except Exception:
        pass
    if i % 10 == 0 or i == len(tickers_alvo):
        print(f'  ✔ {i}/{len(tickers_alvo)}')

df_bt = pd.DataFrame(resultados_bt)
print(f'\n✅ Backtest concluído — {len(df_bt)} ações avaliadas\n')

n_comp = 0
if not df_bt.empty:
    compras_bt = (df_bt[df_bt['Sugestão (3d atrás)'] == '✅ COMPRAR'].copy())
    n_comp = len(compras_bt)
    if n_comp > 0:
        # Δ ajustado p/ ranquear (mesma convicção da Célula 4, versão compacta)
        compras_bt['Δ Ajustado (%)'] = (compras_bt['Δ Previsto 3d (%)']
                                        * compras_bt['LCB Acerto (%)'] / 50.0
                                        * compras_bt['Consenso (%)'] / 100.0
                                        * compras_bt['Sinal/Ruído'].clip(upper=2.0) / 2.0)
        compras_bt = compras_bt.sort_values('Δ Ajustado (%)', ascending=False)
        cart_bt = compras_bt.head(TOP_N).copy()
        w = pesos_conviccao(cart_bt['Δ Ajustado (%)'].values)
        cart_bt['Peso (%)'] = w

        ret_liq_acao = cart_bt['Δ Executado (%)'].values - CUSTO_OPER
        ret_cart  = float((ret_liq_acao * w / 100).sum())
        ret_igual = float((compras_bt['Δ Executado (%)'] - CUSTO_OPER).mean())
        acertos = int((compras_bt['Acertou Direção?'] == '✅ SIM').sum())

        print('═' * 68)
        print(f'  💰 SIMULAÇÃO — R$ {INVESTIMENTO:,.0f} | entrada na abertura, custo de '
              f'{CUSTO_OPER:.1f}% já descontado')
        print('═' * 68)
        print(f'  Ações sugeridas             : {n_comp} (carteira usa top {len(cart_bt)})')
        print(f'  Acerto de direção           : {acertos}/{n_comp} ({acertos/n_comp*100:.1f}%)')
        print(f'  Selic {HORIZONTE}d ref.              : {SELIC_3D:+.3f}%')
        print('  ──────────────────────────────────────────────')
        print(f'  (a) Peso igual (líquido)    : {ret_igual:+.2f}%  →  R$ {INVESTIMENTO*(1+ret_igual/100):,.2f}')
        print(f'  (b) Carteira Convicção (líq): {ret_cart:+.2f}%  →  R$ {INVESTIMENTO*(1+ret_cart/100):,.2f}')
        print(f'  Bateu Selic {HORIZONTE}d?             : '
              f'{"✅ SIM" if max(ret_igual, ret_cart) > SELIC_3D else "❌ NÃO"}')
        print('═' * 68)
        cols_bt = ['Ticker', 'Modelo Campeão', 'Δ Previsto 3d (%)', 'Δ Executado (%)',
                   'Consenso (%)', 'Sinal/Ruído', 'Acertou Direção?']
        print('\n── Sugestões do backtest e o que aconteceu ──')
        print(compras_bt[cols_bt].to_string(index=False))
    else:
        print('⚠️ Nenhuma ação passou no funil neste backtest — capital teria ficado '
              'em caixa rendendo Selic. (Com os filtros da v2 isso é esperado em '
              'parte dos dias — sinal fraco não vira ordem.)')

    excel_bt = f'{PASTA_BT}/Midas_Lab_3D_v3_Backtest.xlsx'
    with pd.ExcelWriter(excel_bt, engine='openpyxl') as writer:
        df_bt.to_excel(writer, sheet_name='Backtest Completo', index=False)
        if n_comp > 0:
            compras_bt.to_excel(writer, sheet_name='Sugestões Compra', index=False)
            cart_bt.to_excel(writer, sheet_name='Carteira Convicção', index=False)
        estilizar_excel(writer, col_sugestao='Sugestão (3d atrás)')
    print(f'\n📊 Excel backtest salvo em: {excel_bt}')


---
## 🔁 Célula 8 — Prova Real: 30 janelas de 3 dias, agora com dinheiro de verdade (simulado direito)

A régua definitiva da v2. Em cada uma das 30 janelas:
- campeão escolhido **só com dados anteriores** (LCB, 20 janelas de validação);
- funil completo (threshold → classe → consenso → ruído → **regime IBOV da época**);
- entrada na **abertura do 1º dia** da janela, custo descontado;
- janelas sem sinal ficam em **caixa rendendo Selic** — na v2 isso acontece mais, e está certo.

Métricas: retorno médio, **Sharpe**, **máximo drawdown** do saldo e **profit factor** (soma dos ganhos ÷ soma das perdas). Sharpe alto + drawdown baixo + profit factor > 1,3 = estratégia com cara de real.


In [ ]:
# ============================================================
# CÉLULA 8 — PROVA REAL v3: 30 JANELAS + SHARPE + DRAWDOWN + PF
# ============================================================

PASTA_PR = 'prova_real_3d'
os.makedirs(PASTA_PR, exist_ok=True)
INVESTIMENTO = 100_000
# N_PROVA e N_VAL_PR agora vêm da Célula 1 (o corte do LSTM depende deles)

registros_pr, resumo_janelas = [], []
print(f'Prova real: {N_PROVA} janelas de {HORIZONTE} pregões × {len(tickers_alvo)} ações...\n')

for w in range(N_PROVA, 0, -1):
    candidatos = []
    for tk in tickers_alvo:
        try:
            df_tk  = cache_precos[tk]
            closes = df_tk['Close'].values.astype(float)
            opens  = df_tk['Open'].values.astype(float)
            datas  = df_tk['Data']
            n = len(closes)
            corte = n - HORIZONTE * w
            alvo_idx = corte - 1 + HORIZONTE
            if corte < MELHOR_JANELA + HORIZONTE * N_VAL_PR or alvo_idx > n - 1:
                continue

            passado = closes[:corte]
            conf_tk = LSTM_CONF.get(tk)
            conf_pass = conf_tk[:corte] if conf_tk is not None and len(conf_tk) >= corte else None
            cval_pr = (float(conf_pass[-1]) if conf_pass is not None and len(conf_pass)
                       and np.isfinite(conf_pass[-1]) else None)
            camp = escolher_campeao(passado, MELHOR_JANELA, n_val=N_VAL_PR, conf=conf_pass)
            if camp is None:
                continue
            classe = classificar(camp['lcb'])

            p0    = float(passado[-1])
            saida = float(closes[alvo_idx])
            treino = passado[-MELHOR_JANELA:]
            if camp['modelo'] == 'LSTM Global' and cval_pr is not None:
                pred = float(preco_de_confianca(p0, cval_pr, treino, HORIZONTE))
            else:
                pred = float(prever(camp['modelo'], treino, HORIZONTE))
            delta_prev = (pred - p0) / p0 * 100
            delta_real = (saida - p0) / p0 * 100
            entrada_exec = float(opens[corte]) if ENTRADA_ABERTURA else p0
            delta_exec = (saida - entrada_exec) / entrada_exec * 100

            vol3d    = vol_movimento_h(treino)
            snr      = abs(delta_prev) / vol3d if vol3d > 0 else 0.0
            consenso = consenso_direcional(treino, np.sign(pred - p0), conf_val=cval_pr)
            sug = sugerir(classe, delta_prev, consenso, snr,
                          regime_ok=regime_em(datas.iloc[corte - 1]))

            candidatos.append({
                'Janela'          : f'J-{w:02d}',
                'Entrada'         : str(datas.iloc[corte - 1].date()),
                'Saída'           : str(datas.iloc[alvo_idx].date()),
                'Ticker'          : tk.replace('.SA', ''),
                'Modelo Campeão'  : camp['modelo'],
                'LCB (%)'         : round(camp['lcb'], 1),
                'Consenso (%)'    : round(consenso, 0),
                'Sinal/Ruído'     : round(snr, 2),
                'Δ Previsto (%)'  : round(delta_prev, 2),
                'Δ Real fech (%)' : round(delta_real, 2),
                'Δ Executado (%)' : round(delta_exec, 2),
                'Δ Ajustado (%)'  : round(delta_prev * camp['lcb']/50.0
                                          * consenso/100.0 * min(snr, 2.0)/2.0, 2),
                'Acertou Direção?': '✅ SIM' if np.sign(delta_prev) == np.sign(delta_real) else '❌ NÃO',
                'Sugestão'        : sug,
            })
        except Exception:
            pass

    df_j = pd.DataFrame(candidatos)
    compras_j = (df_j[df_j['Sugestão'] == '✅ COMPRAR']
                 .sort_values('Δ Ajustado (%)', ascending=False)
                 if not df_j.empty else pd.DataFrame())
    n_comp = len(compras_j)

    if n_comp > 0:
        cart = compras_j.head(TOP_N)
        wgt = pesos_conviccao(cart['Δ Ajustado (%)'].values)
        ret_cart  = float(((cart['Δ Executado (%)'].values - CUSTO_OPER) * wgt / 100).sum())
        ret_igual = float((compras_j['Δ Executado (%)'] - CUSTO_OPER).mean())
        acerto_j  = float((compras_j['Acertou Direção?'] == '✅ SIM').mean() * 100)
        situacao  = 'INVESTIDO'
    else:
        ret_cart = ret_igual = SELIC_3D
        acerto_j, situacao = np.nan, 'CAIXA (Selic)'

    registros_pr.extend(candidatos)
    resumo_janelas.append({
        'Janela'              : f'J-{w:02d}',
        'Período'             : (f"{df_j['Entrada'].min()} → {df_j['Saída'].max()}"
                                 if not df_j.empty else '—'),
        'Situação'            : situacao,
        'Sugestões'           : n_comp,
        'Retorno Igual (%)'   : round(ret_igual, 2),
        'Retorno Carteira (%)': round(ret_cart, 2),
        'Acerto Dir. (%)'     : round(acerto_j, 1) if n_comp > 0 else None,
    })
    print(f'  J-{w:02d} | sugestões: {n_comp:3d} | carteira: {ret_cart:+6.2f}% | '
          f'igual: {ret_igual:+6.2f}% | {situacao}')

df_resumo_pr = pd.DataFrame(resumo_janelas)
df_reg_pr    = pd.DataFrame(registros_pr)

# ── métricas globais ─────────────────────────────────────────
rets_cart  = df_resumo_pr['Retorno Carteira (%)'].astype(float).values
rets_igual = df_resumo_pr['Retorno Igual (%)'].astype(float).values

saldo, saldos = float(INVESTIMENTO), []
for r in rets_cart:
    saldo *= (1 + r / 100)
    saldos.append(round(saldo, 2))
df_resumo_pr['Saldo Acum. (R$)'] = saldos

curva = np.array([INVESTIMENTO] + saldos)
picos = np.maximum.accumulate(curva)
max_dd = float(((curva - picos) / picos).min() * 100)          # máximo drawdown (%)

jan_inv = df_resumo_pr['Situação'] == 'INVESTIDO'
rets_inv = rets_cart[jan_inv.values]
ganhos = rets_inv[rets_inv > 0].sum()
perdas = -rets_inv[rets_inv < 0].sum()
profit_factor = float(ganhos / perdas) if perdas > 0 else float('inf')

selic_acum = [INVESTIMENTO * (1 + SELIC_3D/100) ** k for k in range(1, N_PROVA + 1)]
ret_acum  = (saldo / INVESTIMENTO - 1) * 100
ret_selic = (selic_acum[-1] / INVESTIMENTO - 1) * 100

media_cart  = float(np.mean(rets_cart))
desvio_cart = float(np.std(rets_cart, ddof=1)) if len(rets_cart) > 1 else 0.0
sharpe      = (media_cart - SELIC_3D) / desvio_cart if desvio_cart > 0 else float('nan')
sharpe_anual = sharpe * np.sqrt(252 / HORIZONTE) if desvio_cart > 0 else float('nan')
acerto_med  = float(df_resumo_pr.loc[jan_inv, 'Acerto Dir. (%)'].astype(float).mean()) if jan_inv.any() else float('nan')
n_bateu     = int((rets_cart > SELIC_3D).sum())

print('\n' + '═' * 72)
print(f'  ⚗  PROVA REAL v3 — {N_PROVA} JANELAS | execução realista, custo descontado')
print('═' * 72)
print(f'  Retorno médio/janela (carteira): {media_cart:+.2f}%   (peso igual: {np.mean(rets_igual):+.2f}%)')
print(f'  Selic {HORIZONTE}d de referência        : {SELIC_3D:+.3f}%')
print(f'  Janelas acima da Selic         : {n_bateu}/{N_PROVA} | investidas: {int(jan_inv.sum())}/{N_PROVA}')
print(f'  Acerto direcional médio        : {acerto_med:.1f}%')
print(f'  ★ Sharpe por janela            : {sharpe:.2f}  (anualizado ≈ {sharpe_anual:.2f})')
print(f'  ★ Máximo drawdown              : {max_dd:.2f}%')
print(f'  ★ Profit factor                : {profit_factor:.2f}')
print(f'  Saldo final                    : R$ {saldo:,.2f}  ({ret_acum:+.2f}%)')
print(f'  Selic acumulada                : R$ {selic_acum[-1]:,.2f}  ({ret_selic:+.2f}%)')
print(f'  Alpha vs Selic                 : {ret_acum - ret_selic:+.2f} p.p.')
print('═' * 72)

# ── gráfico compilado 2×2 ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
labels_j = df_resumo_pr['Janela']

ax1 = axes[0, 0]
cores_r = ['#00FF88' if v > SELIC_3D else '#FF6B6B' for v in rets_cart]
ax1.bar(labels_j, rets_cart, color=cores_r, edgecolor='#333333')
ax1.axhline(SELIC_3D, color=COR_OURO, lw=1.5, ls='--', label=f'Selic {HORIZONTE}d ({SELIC_3D:.2f}%)')
ax1.axhline(0, color='#FFFFFF', lw=0.6)
estilo_eixo(ax1, 'Retorno líquido por janela — carteira convicção (%)',
            xlabel='Janela', ylabel='Retorno (%)')
ax1.tick_params(axis='x', rotation=90, labelsize=7)
ax1.legend(facecolor=COR_PAINEL, labelcolor='white', fontsize=8)

ax2 = axes[0, 1]
ax2.plot(labels_j, df_resumo_pr['Saldo Acum. (R$)'], color=COR_OURO, lw=2,
         marker='o', ms=4, label='Midas 3D v2')
ax2.plot(labels_j, selic_acum, color='#AAAAAA', lw=1.5, ls='--', marker='s', ms=3,
         label='Selic acumulada')
ax2.fill_between(range(N_PROVA), df_resumo_pr['Saldo Acum. (R$)'], selic_acum,
                 alpha=0.10, color='#00FF88' if saldo > selic_acum[-1] else '#FF4444')
estilo_eixo(ax2, 'Saldo acumulado vs Selic', xlabel='Janela', ylabel='Saldo (R$)')
ax2.tick_params(axis='x', rotation=90, labelsize=7)
ax2.legend(facecolor=COR_PAINEL, labelcolor='white', fontsize=8)

ax3 = axes[1, 0]
ac_plot = df_resumo_pr['Acerto Dir. (%)'].astype(float).fillna(0).values
cores_a = ['#00FF88' if v >= 56 else COR_OURO if v >= 52 else '#FF6B6B' for v in ac_plot]
ax3.bar(labels_j, ac_plot, color=cores_a, edgecolor='#333333')
ax3.axhline(50, color='#AAAAAA', lw=0.8, ls=':', label='Aleatório (50%)')
estilo_eixo(ax3, 'Acerto direcional por janela (0 = em caixa)',
            xlabel='Janela', ylabel='Acerto (%)')
ax3.tick_params(axis='x', rotation=90, labelsize=7)
ax3.legend(facecolor=COR_PAINEL, labelcolor='white', fontsize=8)

ax4 = axes[1, 1]
ax4.set_facecolor(COR_FUNDO); ax4.axis('off')
linhas_sc = [
    ('⚗ MIDAS 3D v3 — SCORECARD', COR_OURO, 14, 'bold'),
    ('', '#CCCCCC', 8, 'normal'),
    (f'Retorno médio/janela (líquido) : {media_cart:+.2f}%', '#CCCCCC', 11, 'normal'),
    (f'Selic {HORIZONTE}d referência           : {SELIC_3D:+.3f}%', '#AAAAAA', 11, 'normal'),
    (f'★ Sharpe: {sharpe:.2f}  |  anualizado ≈ {sharpe_anual:.1f}',
     '#00FF88' if sharpe > 0 else '#FF4444', 11, 'bold'),
    (f'★ Máx. drawdown: {max_dd:.1f}%  |  Profit factor: {profit_factor:.2f}',
     '#CCCCCC', 11, 'normal'),
    ('', '#CCCCCC', 8, 'normal'),
    (f'Acerto direcional médio        : {acerto_med:.1f}%', '#CCCCCC', 11, 'normal'),
    (f'Janelas > Selic: {n_bateu}/{N_PROVA}  |  investidas: {int(jan_inv.sum())}/{N_PROVA}',
     '#CCCCCC', 11, 'normal'),
    ('', '#CCCCCC', 8, 'normal'),
    (f'R$ 100k → R$ {saldo:,.0f}',
     '#00FF88' if saldo > INVESTIMENTO else '#FF4444', 13, 'bold'),
    (f'Acumulado: {ret_acum:+.1f}%  (Selic: {ret_selic:+.1f}%)',
     '#00FF88' if ret_acum > ret_selic else '#FF4444', 12, 'bold'),
    ('', '#CCCCCC', 8, 'normal'),
    ('⚠️ Sugestão algorítmica. Não constitui', '#666666', 8, 'italic'),
    ('   recomendação de investimento.', '#666666', 8, 'italic'),
]
y_pos = 0.97
for texto, cor, tam, peso in linhas_sc:
    ax4.text(0.05, y_pos, texto, transform=ax4.transAxes, color=cor, fontsize=tam,
             fontweight=peso if peso != 'italic' else 'normal',
             fontstyle='italic' if peso == 'italic' else 'normal', va='top')
    y_pos -= 0.062

fig.patch.set_facecolor(COR_FUNDO)
fig.suptitle(f'Midas 3D v3 — Prova Real: {N_PROVA} janelas  |  Sharpe {sharpe:.2f}  |  '
             f'DD {max_dd:.1f}%  |  Acumulado {ret_acum:+.1f}%',
             color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{PASTA_PR}/_compilado_30_janelas.png', dpi=120,
            bbox_inches='tight', facecolor=COR_FUNDO)
plt.show()

# ── Excel compilado ───────────────────────────────────────────
excel_pr = f'{PASTA_PR}/Midas_Lab_3D_v3_ProvaReal.xlsx'
metricas_df = pd.DataFrame([
    {'Métrica': 'Retorno médio/janela — carteira líquida (%)', 'Valor': round(media_cart, 3)},
    {'Métrica': 'Retorno médio/janela — peso igual líquido (%)','Valor': round(float(np.mean(rets_igual)), 3)},
    {'Métrica': f'Selic {HORIZONTE}d de referência (%)',        'Valor': round(SELIC_3D, 3)},
    {'Métrica': 'Sharpe por janela (excedente à Selic)',        'Valor': round(sharpe, 3)},
    {'Métrica': 'Sharpe anualizado (aprox.)',                   'Valor': round(sharpe_anual, 2)},
    {'Métrica': 'Máximo drawdown (%)',                          'Valor': round(max_dd, 2)},
    {'Métrica': 'Profit factor (janelas investidas)',           'Valor': (round(profit_factor, 2)
                                                                          if np.isfinite(profit_factor) else 'inf')},
    {'Métrica': 'Acerto direcional médio (%)',                  'Valor': round(acerto_med, 1)},
    {'Métrica': 'Janelas acima da Selic',                       'Valor': f'{n_bateu}/{N_PROVA}'},
    {'Métrica': 'Saldo final (R$)',                             'Valor': round(saldo, 2)},
    {'Métrica': 'Retorno acumulado (%)',                        'Valor': round(ret_acum, 2)},
    {'Métrica': 'Alpha vs Selic (p.p.)',                        'Valor': round(ret_acum - ret_selic, 2)},
])
with pd.ExcelWriter(excel_pr, engine='openpyxl') as writer:
    df_resumo_pr.to_excel(writer, sheet_name='Resumo 30 Janelas', index=False)
    if not df_reg_pr.empty:
        df_reg_pr.to_excel(writer, sheet_name='Registros', index=False)
    metricas_df.to_excel(writer, sheet_name='Métricas Globais', index=False)
    estilizar_excel(writer)
print(f'\n📊 Compilado salvo em : {excel_pr}')
print(f'🖼️  Gráfico salvo em  : {PASTA_PR}/_compilado_30_janelas.png')


---
## 📋 Célula 9 — Como ler a v3 (e o que ela sacrifica de propósito)

**A régua, na ordem:**
1. **Sharpe + drawdown + profit factor** (Célula 6). Regra de bolso: Sharpe/janela > 0,3 com drawdown < 10% e PF > 1,3 = estratégia respirando; Sharpe ≈ 0 = as sugestões do dia são ruído bonito.
2. **LCB, não acerto bruto.** Um "68% de acerto" com LCB 45% é sorte estatística; a v2 já ranqueia assim, mas vale internalizar.
3. **OBSERVAR é informação**: passou no retorno e na classe, falhou em consenso/ruído/regime. É a fila de espera — se aparecer de novo amanhã com consenso maior, subiu de convicção.
4. **Dias sem COMPRAR são feature, não bug.** O funil corta sinal fraco; em regime de baixa do IBOV, corta quase tudo. Caixa rendendo Selic é uma posição.

**O que a v2 sacrifica de propósito:** quantidade de sinais e retorno "de papel". Entrada na abertura + custo descontado + filtros de convicção derrubam os números bonitos da v1 — porque os da v1 eram parcialmente inatingíveis. Compare as duas Provas Reais e confie na menor.

**Sobre os modelos novos:** se o ARIMA ou o LSTM não ganharem o campeonato, isso é resultado, não defeito — em 3 dias, métodos simples frequentemente empatam ou vencem redes neurais, e o notebook existe justamente para medir isso na SUA carteira em vez de assumir. A LSTM fica desatualizada com o tempo: re-rode a Célula 4 (retreino completo) pelo menos 1x por mês. E lembre que a acurácia de validação dela é pré-corte; a nota que vale é a do campeonato (LCB).

**Riscos que nenhum filtro remove:** fato relevante (RJ, grupamento, diluição) estoura qualquer modelo de preço; spread real pode passar do `CUSTO_OPER` configurado — confira o book antes; e 30 janelas ainda é amostra curta para conclusões definitivas.

> ⚠️ *Sugestão algorítmica para estudo. Não constitui recomendação de investimento.*
